# GVHMR → your avatar: video → SMPL `.pkl` (Colab)

This is the **official GVHMR Colab** + **one extra step**: it exports the SMPL
motion as the `.pkl` your `dan` repo already understands, and downloads it.

**Flow:** upload a dance video → GVHMR extracts SMPL → convert to repo `.pkl` →
download → (locally) `python scripts/video_to_smpl/build_clip.py` → avatar dances it.

Runtime → **Change runtime type → GPU (T4)** before running.

> For most dance tutorials (tripod / fixed camera) use the `-s` flag — it skips
> the slow SLAM/visual-odometry step, so you can even skip the DPVO install.

In [ ]:
!nvidia-smi

## 1. Install GVHMR (~10 min)

In [ ]:
import os
from pathlib import Path

!git clone https://github.com/zju3dv/GVHMR --recursive
proj_root = str(Path('GVHMR').absolute())

%cd {proj_root}
%pip install -r requirements.txt
%pip install -e .

### 1a. Config deps — RUN THIS (fixes `No module named 'hydra'`)
GVHMR's demo needs Facebook **Hydra**, whose PyPI name is `hydra-core` (it
imports as `hydra`). ⚠️ NEVER `pip install hydra` — that is a different,
abandoned package (`Hydra-2.5`) that fails to build a wheel on Python 3.12,
which is the exact error you hit.


In [ ]:
# Full GVHMR runtime dep stack that Colab's drifted image is missing.
# hydra-core -> `import hydra`; hydra-zen -> register_store_gvhmr; smplx -> body model.
# The rest are demo.py imports that GVHMR's requirements.txt didn't pull on this image.
%pip install hydra-core hydra-zen omegaconf colorlog
%pip install smplx einops timm chumpy
%pip install ultralytics opencv-python imageio imageio-ffmpeg av
%pip install joblib scikit-image pycocotools

# verify the imports demo.py actually makes, up front
import hydra, hydra_zen, omegaconf, smplx, einops, timm, ultralytics
print('deps OK | hydra', hydra.__version__, '| smplx', smplx.__version__)


### 1a-2. PyTorch3D — RUN THIS (fixes `No module named 'pytorch3d'`)
Colab now ships torch 2.11 / CUDA 12.8 on Python 3.12, and **no prebuilt
pytorch3d wheel exists** for that combo — it must be **compiled from source
against Colab's installed torch**. The `--no-build-isolation` flag is
essential: without it pip builds against a *different* torch and you get a
silent ABI mismatch. Build takes **~10–20 min** (that's normal, not a hang).


In [ ]:
# pytorch3d has no wheel for Colab's torch 2.11 / py3.12 -> build from source
# against the INSTALLED torch. --no-build-isolation is what makes it link the
# right torch. ninja speeds the compile.
%pip install ninja fvcore iopath
%pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git@stable"
import pytorch3d
from pytorch3d.transforms import quaternion_to_matrix  # the exact import demo.py needs
print('pytorch3d', pytorch3d.__version__, 'OK')


### 1b. DPVO — OPTIONAL, only for moving-camera video
Skip this whole cell if your videos are shot on a fixed tripod (you'll run demo
with `-s`). Installing DPVO is the slowest, most fragile part.

In [ ]:
%cd {proj_root}/third-party/DPVO

!wget https://gitlab.com/libeigen/eigen/-/archive/3.4.0/eigen-3.4.0.zip
!unzip -o eigen-3.4.0.zip -d thirdparty && rm -rf eigen-3.4.0.zip

%pip install torch-scatter -f "https://data.pyg.org/whl/torch-2.3.0+cu121.html"
%pip install numba pypose

if 'cuda_home' not in locals():
  cuda_home = '/usr/local/cuda-12'
  if not Path(cuda_home).exists():
    raise FileNotFoundError('CUDA_HOME for cuda 12.x not found!')
  os.environ['CUDA_HOME'] = cuda_home
  os.environ['PATH'] = os.environ['PATH'] + f':{cuda_home}/bin'

%pip install -e .
%cd {proj_root}

## 2. Download model checkpoints (~1 min)

In [ ]:
%cd {proj_root}
!mkdir -p inputs outputs inputs/demo

!apt install -y -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl -d inputs/checkpoints/body_models/smpl -o SMPL_NEUTRAL.pkl
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPLX_NEUTRAL.npz -d inputs/checkpoints/body_models/smplx -o SMPLX_NEUTRAL.npz
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/dpvo/dpvo.pth -d inputs/checkpoints/dpvo -o dpvo.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/gvhmr/gvhmr_siga24_release.ckpt -d inputs/checkpoints/gvhmr -o gvhmr_siga24_release.ckpt
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/hmr2/epoch%3D10-step%3D25000.ckpt -d inputs/checkpoints/hmr2 -o epoch=10-step=25000.ckpt
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/vitpose/vitpose-h-multi-coco.pth -d inputs/checkpoints/vitpose -o vitpose-h-multi-coco.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/GVHMR/resolve/main/yolo/yolov8x.pt -d inputs/checkpoints/yolo -o yolov8x.pt

## 3. Upload YOUR dance video
Run this and pick a short (5–20s) clip with **one person, full body visible**.

In [ ]:
%cd {proj_root}
from google.colab import files
up = files.upload()
src = list(up.keys())[0]
video_name = Path(src).stem

# ── CRITICAL PRE-PROCESS ──────────────────────────────────────────────
# A free T4 chokes on 4K / long clips (the demo stalls in preprocessing).
# Trim to the first MAX_SECONDS and downscale so the long side <= 720px.
# This one step is what turns "stalls / ^C" into a clean ~2-min run.
MAX_SECONDS = 12          # 8-15s is the sweet spot for a first test
raw = f'inputs/demo/{video_name}_raw.mp4'
os.replace(src, raw)
run_name = f'{video_name}_proc'
video_path = f'inputs/demo/{run_name}.mp4'
!ffmpeg -y -i {raw} -t {MAX_SECONDS} -r 30 \
  -vf "scale='if(gt(iw,ih),720,-2)':'if(gt(iw,ih),-2,720)'" \
  -an {video_path}

import subprocess
info = subprocess.run(['ffprobe','-v','error','-select_streams','v:0',
    '-count_frames','-show_entries','stream=nb_read_frames,width,height',
    '-of','csv=p=0', video_path], capture_output=True, text=True).stdout.strip()
print('run_name   =', run_name)
print('video_path =', video_path, '| WxH,frames =', info)


## 4. Run GVHMR
`-s` = static camera (recommended for tripod dance videos). Remove `-s` only if
the camera moves (and you installed DPVO in 1b).

In [ ]:
!python {proj_root}/tools/demo/demo.py --video={proj_root}/{video_path} -s

### Sanity check #1 — does the GVHMR mesh track the dancer?
If the mesh follows the person in this render, the SMPL is good BEFORE we ever
touch your avatar.

In [ ]:
import io, base64, os
from IPython.display import HTML
from hmr4d.utils.video_io_utils import get_video_lwh

# The SMPL is saved to hmr4d_results.pt BEFORE the preview render runs, so a
# render crash (e.g. chumpy vs py3.12) does NOT lose your motion — you can skip
# straight to the convert cell. This just shows the overlay if it exists.
def display_video(fn):
    L, W, H = get_video_lwh(fn)
    scale = min(W, 1080) / W
    W, H = int(W * scale), int(H * scale)
    enc = base64.b64encode(io.open(fn, 'rb').read())
    return HTML(data='<video width="{0}" height="{1}" controls>'
        '<source src="data:video/mp4;base64,{2}" type="video/mp4" /></video>'
        .format(W, H, enc.decode('ascii')))

pt = f'outputs/demo/{run_name}/hmr4d_results.pt'
print('SMPL saved :', os.path.exists(pt), '->', pt)
overlay = f'outputs/demo/{run_name}/{run_name}_3_incam_global_horiz.mp4'
display_video(overlay) if os.path.exists(overlay) else print(
    'no overlay render (fine — SMPL above is what matters, go to convert cell)')


## 5. ⭐ Convert GVHMR output → your repo's SMPL `.pkl`
This is the ONLY thing the official Colab is missing. It reads
`hmr4d_results.pt` (SMPL-X, gravity-aligned world frame) and writes the exact
`{smpl_poses (T,72), smpl_trans (T,3), smpl_scaling, fps}` schema your
`load_aist_pkl` / `export_motion_json.py` already ingest.

This is the inline twin of `scripts/video_to_smpl/gvhmr_to_aist.py`.

In [ ]:
import torch, numpy as np, pickle

FPS = 30.0   # GVHMR demo resamples to 30 fps

def _np(x):
    return x.detach().cpu().numpy() if hasattr(x, 'detach') else np.asarray(x)

def gvhmr_pt_to_aist(pt_path, out_pkl, fps=FPS):
    pred = torch.load(pt_path, map_location='cpu')
    assert 'smpl_params_global' in pred, list(pred.keys())
    p = pred['smpl_params_global']
    go = _np(p['global_orient']).reshape(-1, 3)          # (T,3) axis-angle pelvis
    bp = _np(p['body_pose']).reshape(len(go), -1)        # (T,63) SMPL-X body
    tr = _np(p['transl']).reshape(-1, 3)                 # (T,3) meters, world
    T = len(go)
    # SMPL-X body_pose (63 = joints 1..21) -> SMPL 69 by padding hands (22,23)
    body = np.concatenate([bp, np.zeros((T, 6))], axis=1) if bp.shape[1] == 63 else bp
    smpl_poses = np.concatenate([go, body], axis=1).astype(np.float64)   # (T,72)
    smpl_trans = tr.astype(np.float64)
    smpl_trans = smpl_trans - smpl_trans[0:1] * np.array([1., 0., 1.])   # center XZ
    out = {'smpl_poses': smpl_poses, 'smpl_trans': smpl_trans,
           'smpl_scaling': 1.0, 'fps': float(fps)}
    with open(out_pkl, 'wb') as f:
        pickle.dump(out, f)
    print('wrote', out_pkl)
    print('  frames =', T, ' fps =', fps,
          ' finite =', np.isfinite(smpl_poses).all() and np.isfinite(smpl_trans).all())
    print('  pose range [%.3f, %.3f] rad' % (smpl_poses.min(), smpl_poses.max()))
    print('  height Y range [%.3f, %.3f] m' % (smpl_trans[:,1].min(), smpl_trans[:,1].max()))
    return out

pt_path = f'outputs/demo/{run_name}/hmr4d_results.pt'
out_pkl = f'{run_name}.pkl'
gvhmr_pt_to_aist(pt_path, out_pkl)


## 6. Download the `.pkl`

In [ ]:
from google.colab import files
files.download(out_pkl)

## 7. Locally (in `c:\dan`) — put it on your avatar

Download `<run_name>.pkl`, then retarget + render on the VRM avatar
(use the Python that has the render deps — cv2/pyrender/trimesh/mediapipe):

```powershell
$py = "C:\Users\rohitacharya\AppData\Local\Programs\Python\Python311\python.exe"

# 1) retarget SMPL -> VRM-quat, validate, install into coach/motion_cache
& $py scripts/video_to_smpl/build_clip.py `
    --aist <run_name>.pkl --name <run_name> `
    --vrm data/models/extra/AliciaSolid.vrm `
    --smpl_pkl data/models/smpl_raw/smpl/models/basicmodel_m_lbs_10_207_0_v1.0.0.pkl

# 2) render the FULL clip on the avatar (default caps at 90 frames; pass --frames)
& $py scripts/play_smpl_motion.py `
    --aist <run_name>.pkl --vrm data/models/extra/AliciaSolid.vrm `
    --smpl_pkl data/models/smpl_raw/smpl/models/basicmodel_m_lbs_10_207_0_v1.0.0.pkl `
    --frames 240 --out data/output_videos/<run_name>.mp4
```

- `passed=True` + `coach/motion_cache/<run_name>.json` → success (served at
  `/api/motion/data/<run_name>.json`).
- `data/output_videos/<run_name>.mp4` = your avatar performing the dance.
- Avatars live (hidden attr) in `data/models/extra/*.vrm`
  (AliciaSolid, AvatarSample_K, Theobro, …) and `data/models/*.vrm` (mymodel1/2).

**If the avatar comes out rotated / lying down:** run the downloaded `.pkl`
through `gvhmr_to_aist.py` locally with `--root-fix x-90` (or `x+90` / `y180`) —
a cheap coordinate-frame retry.
